This notebook collects current-season data from FBref (FBref, n.d.) using the SoccerData library (SoccerData developers, n.d.) for the application.

It merges the FBref statistics with local Transfermarkt-derived player information and latest recorded market values.

The resulting dataset lets the application select the appropriate model according to whether a player has a previous market value.

Flow: FBref current data → club context → Transfermarkt identity/static data → one player-season row → end-of-season projection → model-ready features.

The output is a projected 2026/27 player-season dataset for a target valuation date of 2027-06-01.

### 1. Imports and Configuration

In [102]:
import re

import numpy as np
import pandas as pd
import soccerdata as sd
from unidecode import unidecode

SEASON = 2026
FBREF_SEASON = "2026-27"
SNAPSHOT_DATE = pd.Timestamp("2026-10-03")
TARGET_VALUATION_DATE = pd.Timestamp("2027-06-01")

LEAGUE_GAMES = {
    "ENG-Premier League": 38,
    "ESP-La Liga": 38,
    "ITA-Serie A": 38,
    "GER-Bundesliga": 34,
    "FRA-Ligue 1": 34,
}

COMPETITION_MAP = {
    "ENG-Premier League": "GB1",
    "ESP-La Liga": "ES1",
    "ITA-Serie A": "IT1",
    "GER-Bundesliga": "L1",
    "FRA-Ligue 1": "FR1",
}


### 2. Fetch Current FBRef Data Using Soccerdata

We will use three different sources from SoccerData library on FBRef data. One for player statistics, one for goalkeeper statistics and one for club-level statistics. Aiming to replicate the features used during modeling.

For more information on the API calls done go to: https://soccerdata.readthedocs.io/en/latest/reference/fbref.html#


In [103]:
# initialise FBref reader for current Big-5 season
fbref = sd.FBref(
    leagues="Big 5 European Leagues Combined",
    seasons="2026-27"
)

# retrieve standard player season statistics
player_stats = fbref.read_player_season_stats(
    stat_type="standard"
)
# retrieve keeper stats for goalkeepers
keeper_stats = fbref.read_player_season_stats(
    stat_type="keeper"
)
# retrieve the match schedule for match-level results
schedule = fbref.read_schedule()

[10/5/2026 10:21:11 PM] INFO     Saving cached data to C:\Users\Ignacio Castro\soccerdata\data\FBref _common.py:250

[10/5/2026 10:21:13 PM] WARNING  C:\Users\Ignacio                                                   ]8;id=16692274;file://c:\ProgramData\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=16692275;file://c:\ProgramData\anaconda3\Lib\warnings.py#110\110]8;;\
                                 Castro\AppData\Roaming\Python\Python313\site-packages\soccerdata\f                
                                 bref.py:171: FutureWarning: The behavior of DataFrame                             
                                 concatenation with empty or all-NA entries is deprecated. In a                    
                                 future version, this will no longer exclude empty or all-NA                       
                                 columns when determining the result dtypes. To retain the old                     
                                 behavior, exclude the relevant entries before the concat                          
                                 operation.                                                                        
                                   pd.concat(dfs)                                                                  
                                                                                                                   

[10/5/2026 10:21:15 PM] WARNING  C:\Users\Ignacio                                                   ]8;id=16692280;file://c:\ProgramData\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=16692281;file://c:\ProgramData\anaconda3\Lib\warnings.py#110\110]8;;\
                                 Castro\AppData\Roaming\Python\Python313\site-packages\soccerdata\f                
                                 bref.py:171: FutureWarning: The behavior of DataFrame                             
                                 concatenation with empty or all-NA entries is deprecated. In a                    
                                 future version, this will no longer exclude empty or all-NA                       
                                 columns when determining the result dtypes. To retain the old                     
                                 behavior, exclude the relevant entries before the concat                          
                                 operation.                                                                        
                                   pd.concat(dfs)                                                                  
                                                                                                                   

                        WARNING  C:\Users\Ignacio                                                   ]8;id=16692286;file://c:\ProgramData\anaconda3\Lib\warnings.py\warnings.py]8;;\:]8;id=16692287;file://c:\ProgramData\anaconda3\Lib\warnings.py#110\110]8;;\
                                 Castro\AppData\Roaming\Python\Python313\site-packages\soccerdata\f                
                                 bref.py:171: FutureWarning: The behavior of DataFrame                             
                                 concatenation with empty or all-NA entries is deprecated. In a                    
                                 future version, this will no longer exclude empty or all-NA                       
                                 columns when determining the result dtypes. To retain the old                     
                                 behavior, exclude the relevant entries before the concat                          
                                 operation.                                                                        
                                   pd.concat(dfs)                                                                  
                                                                                                                   

### 3. Clean and Standardize FBref Data

Tables need to be cleaned and standardized to make later joins a feature selection consistent.

In this section, we also input missing leagues. During development it was found that the data for Bundesliga (Germany) had the league column identifier empty, for that reason, it was manually imputed.

In [104]:
# define helper func to flatten and clean fbref dfs
def flatten_fbref(df):
    """
    Flatten a pandas DataFrame whose columns are a MultiIndex.

    This function:
    - resets the index into columns,
    - combines tuple-based column labels into a single string,
    - converts column names to lowercase,
    - normalizes spaces, percent symbols, plus signs, and slash separators
      into a consistent, pandas-friendly format.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame with possibly hierarchical MultiIndex columns.

    Returns
    -------
    pandas.DataFrame
        A copy of the DataFrame with flattened, normalized column names.
    """
    df = df.reset_index().copy()

    df.columns = [
        "_".join(
            str(x).strip()
            for x in col
            if str(x).strip()
        )
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]

    df.columns = (
        df.columns
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("%", "pct", regex=False)
        .str.replace("+", "plus", regex=False)
        .str.replace("/", "_per_", regex=False)
    )

    return df

In [105]:
# flatten the different dfs
standard = flatten_fbref(player_stats)
keeper = flatten_fbref(keeper_stats)
schedule_flat = flatten_fbref(schedule)

In [106]:
# check for missing leagues
missing_league_teams = (
    standard.loc[standard["league"].isna(), "team"]
    .drop_duplicates()
)

print(len(missing_league_teams))
print(sorted(missing_league_teams))

18
['Augsburg', 'Bayern Munich', 'Dortmund', 'Elversberg', 'Frankfurt', 'Freiburg', 'Gladbach', 'Hamburger SV', 'Hoffenheim', 'Köln', 'Leverkusen', 'Mainz 05', 'Paderborn 07', 'RB Leipzig', 'Schalke 04', 'Stuttgart', 'Union Berlin', 'Werder Bremen']


In [107]:
# manually input missing league identifier
for df in [standard, keeper]:
    df.loc[df["league"].isna(), "league"] = "GER-Bundesliga"

### 4. Build Current Club Statistics

Club features are calculated from completed league matches. Player /90 metrics are taken directly from FBref.

In [108]:
# take only completed matches
completed_matches = schedule_flat[
    schedule_flat["score"].notna()
].copy()

In [109]:
# extract home and away goals from completed matches
scores = completed_matches["score"].str.extract(
    r"(\d+)\D+(\d+)"
)

completed_matches["home_goals"] = pd.to_numeric(
    scores[0]
)

completed_matches["away_goals"] = pd.to_numeric(
    scores[1]
)

In [110]:
# create home and away team rows
home = pd.DataFrame({
    "team": completed_matches["home_team"],
    "league": completed_matches["league"],
    "games": 1,
    "goals_scored": completed_matches["home_goals"],
    "goals_conceded": completed_matches["away_goals"],
})

away = pd.DataFrame({
    "team": completed_matches["away_team"],
    "league": completed_matches["league"],
    "games": 1,
    "goals_scored": completed_matches["away_goals"],
    "goals_conceded": completed_matches["home_goals"],
})

In [111]:
# create wins, draws and losses features
home["wins"] = (
    completed_matches["home_goals"]
    > completed_matches["away_goals"]
).astype(int)

home["draws"] = (
    completed_matches["home_goals"]
    == completed_matches["away_goals"]
).astype(int)

home["losses"] = (
    completed_matches["home_goals"]
    < completed_matches["away_goals"]
).astype(int)

home["clean_sheets"] = (
    completed_matches["away_goals"] == 0
).astype(int)

away["wins"] = (
    completed_matches["away_goals"]
    > completed_matches["home_goals"]
).astype(int)

away["draws"] = (
    completed_matches["away_goals"]
    == completed_matches["home_goals"]
).astype(int)

away["losses"] = (
    completed_matches["away_goals"]
    < completed_matches["home_goals"]
).astype(int)

away["clean_sheets"] = (
    completed_matches["home_goals"] == 0
).astype(int)

In [112]:
# combine home and away rows
team_matches = pd.concat(
    [home, away],
    ignore_index=True
)

# generate club statistics grouped
club_stats = (
    pd.concat([home, away], ignore_index=True)
    .groupby(["league", "team"], as_index=False)
    .agg(
        club_games_played=("games", "sum"),
        club_goals_scored=("goals_scored", "sum"),
        club_goals_conceded=("goals_conceded", "sum"),
        wins=("wins", "sum"),
        draws=("draws", "sum"),
        losses=("losses", "sum"),
        club_clean_sheets=("clean_sheets", "sum"),
    )
)

In [113]:
# create points and avg_point_per_game features
club_stats["points"] = (
    club_stats["wins"] * 3
    + club_stats["draws"]
)

club_stats["avg_points_per_game"] = (
    club_stats["points"]
    / club_stats["club_games_played"]
)

club_stats["club_clean_sheet_rate"] = (
    club_stats["club_clean_sheets"]
    / club_stats["club_games_played"]
)

display(
    club_stats.sort_values(
        "points",
        ascending=False
    )
)

,league,team,club_games_played,club_goals_scored,club_goals_conceded,wins,draws,losses,club_clean_sheets,points,avg_points_per_game,club_clean_sheet_rate
23,ESP-La Liga,Barcelona,7,31,7,7,0,0,3,21,3.000000,0.428571
34,ESP-La Liga,Real Betis,7,9,7,5,1,1,4,16,2.285714,0.571429
22,ESP-La Liga,Atlético Madrid,7,16,7,5,1,1,3,16,2.285714,0.428571
35,ESP-La Liga,Real Madrid,7,18,8,5,0,2,1,15,2.142857,0.142857
14,ENG-Premier League,Manchester City,5,13,5,5,0,0,2,15,3.000000,0.400000
...,...,...,...,...,...,...,...,...,...,...,...,...
43,FRA-Ligue 1,Le Havre,5,4,7,0,2,3,1,2,0.400000,0.200000
82,ITA-Serie A,Genoa,5,3,10,0,1,4,0,1,0.200000,0.000000
74,GER-Bundesliga,Union Berlin,4,4,17,0,1,3,0,1,0.250000,0.000000
64,GER-Bundesliga,Gladbach,4,6,16,0,0,4,0,0,0.000000,0.000000


In [114]:
# renaming and cleaning
player_current = standard.rename(
    columns={
        "playing_time_mp": "current_games",
        "playing_time_min": "current_minutes",
        "playing_time_90s": "current_90s",
        "performance_gls": "current_goals",
        "performance_ast": "current_assists",
        "performance_crdy": "current_yellow_cards",
        "performance_crdr": "current_red_cards",
        "per_90_minutes_gls": "current_goals_per_90",
        "per_90_minutes_ast": "current_assists_per_90",
        "per_90_minutes_gplusa": "current_goal_contributions_per_90",
    }
)[
    [
        "league",
        "season",
        "team",
        "player",
        "pos",
        "age",
        "born",
        "current_games",
        "current_minutes",
        "current_90s",
        "current_goals",
        "current_assists",
        "current_goals_per_90",
        "current_assists_per_90",
        "current_goal_contributions_per_90",
        "current_yellow_cards",
        "current_red_cards",
    ]
].copy()


keeper_current = keeper.rename(
    columns={"performance_cs": "current_clean_sheets_gk"}
)[
    [
        "league",
        "season",
        "team",
        "player",
        "current_clean_sheets_gk",
    ]
]

# merge player and keeper datasets
player_current = player_current.merge(
    keeper_current,
    on=["league", "season", "team", "player"],
    how="left",
    validate="one_to_one",
)

# merge club stats
fbref_current = player_current.merge(
    club_stats,
    on=["league", "team"],
    how="left",
    validate="many_to_one",
)

#map competition id to transfermarkt code
fbref_current["competition_id"] = fbref_current["league"].map(COMPETITION_MAP)

### 5. Map FBref Players to Transfermarkt

Transfermarkt and FBRef data have different formats, for that reason the following actions were taken:

- player_code_fb is only used to resolve identity. The durable join key after mapping is Transfermarkt player_id.

- Automatic matching is restricted to Transfermarkt player codes that are unique. Ambiguous or unmatched players are resolved through the manual mapping file.

In [115]:
SPECIAL_TRANSLITERATION = str.maketrans({
    "Đ": "Dj",
    "đ": "dj",
    "Ð": "Dj",
    "ð": "dj",
    "Ł": "L",
    "ł": "l",
    "Ø": "O",
    "ø": "o",
    "Æ": "Ae",
    "æ": "ae",
    "Œ": "Oe",
    "œ": "oe",
    "ß": "ss",
})


def create_tm_player_code(name):
    if pd.isna(name):
        return None

    # handle characters where Transfermarkt-style transliteration
    # differs from basic accent removal
    name = name.translate(SPECIAL_TRANSLITERATION)

    # Remove remaining accents / special characters
    name = unidecode(name)

    # Convert to Transfermarkt-style slug
    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9]+", "-", name)
    name = name.strip("-")

    return name


# apply function
fbref_current["player_code_fb"] = (
    fbref_current["player"].apply(create_tm_player_code)
)

In [116]:
tm_players = pd.read_csv("../data/raw/players.csv")
display(tm_players)

tm_players = tm_players[
    [
        "name",
        "player_id",
        "player_code",
        "date_of_birth",
        "position",
        "sub_position",
        "foot",
        "height_in_cm",
        "image_url",
        "country_of_citizenship"

    ]
].drop_duplicates("player_id")


# auto-match player codes that identify one Transfermarkt player
tm_code_counts = tm_players.groupby("player_code")["player_id"].nunique()

tm_unique_codes = tm_players.loc[
    tm_players["player_code"].map(tm_code_counts).eq(1),
    ["player_code", "player_id"],
].rename(columns={"player_id": "player_id_auto"})

fbref_players = (
    fbref_current[["player", "player_code_fb"]]
    .drop_duplicates("player_code_fb")
)

auto_mapping = fbref_players.merge(
    tm_unique_codes,
    left_on="player_code_fb",
    right_on="player_code",
    how="left",
    validate="one_to_one",
)

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus FC,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,Bayern Munich,100000.0,3000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50144,1546617,Yahya Mousa,Gharawi,Yahya Mousa Gharawi,2025,34911,yahya-mousa-gharawi,NaN,NaN,Saudi Arabia,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/yahya-mousa-gh...,SA1,Neom Sports Club,NaN,NaN
50145,1547421,Szabolcs,Nyitra,Szabolcs Nyitra,2025,37255,szabolcs-nyitra,NaN,NaN,Romania,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/szabolcs-nyitr...,RO1,FK Csikszereda Miercurea Ciuc,NaN,NaN
50146,1548905,Iustin,Filote,Iustin Filote,2025,8818,iustin-filote,NaN,NaN,Romania,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/iustin-filote/...,RO1,FC Botosani,NaN,NaN
50147,1550699,Vukasin,Zdravkovic,Vukasin Zdravkovic,2025,7567,vukasin-zdravkovic,NaN,NaN,Serbia,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/vukasin-zdravk...,SER1,FK Radnicki Nis,NaN,NaN


In [117]:
# take manual mapping for missing cases
manual_mapping = pd.read_csv("../data/processed/manual_mapping.csv").rename(
    columns={"player_id_tf": "player_id_manual"}
)

manual_mapping = (
    manual_mapping[["player_code_fb", "player_id_manual"]]
    .drop_duplicates("player_code_fb")
)

manual_mapping["player_id_manual"] = pd.to_numeric(
    manual_mapping["player_id_manual"], errors="coerce"
).astype("Int64")

final_mapping = auto_mapping.merge(
    manual_mapping,
    on="player_code_fb",
    how="left",
    validate="one_to_one",
)

# manual mapping takes precedence when supplied
final_mapping["player_id"] = (
    final_mapping["player_id_manual"]
    .combine_first(final_mapping["player_id_auto"])
)

final_mapping = final_mapping[
    ["player", "player_code_fb", "player_id"]
]

assert final_mapping["player_code_fb"].is_unique

final_mapping.to_csv(
    "../data/processed/mapping_fbref_transfermarkt.csv",
    index=False,
)



### 6. Enrich With Transfermarkt Player Data and Latest Valuation

The mapped Transfermarkt player is used to add static attributes and the latest available market valuation.

In [118]:
latest_market_values = pd.read_csv(
    "../data/processed/latest_market_values.csv"
)[
    [
        "player_id",
        "latest_valuation_date",
        "latest_market_value_in_eur",
    ]
]

In [119]:
latest_market_values["player_id"] = pd.to_numeric(
    latest_market_values["player_id"], errors="coerce"
).astype("Int64")

assert latest_market_values["player_id"].is_unique

current_players = fbref_current.merge(
    final_mapping[["player_code_fb", "player_id"]],
    on="player_code_fb",
    how="left",
    validate="many_to_one",
)


current_players = current_players.merge(
    tm_players[
        [
            "player_id",
            "position",
            "sub_position",
            "foot",
            "height_in_cm",
            "image_url",
            "country_of_citizenship"
        ]
    ],
    on="player_id",
    how="left",
    validate="many_to_one",
)

current_players = current_players.merge(
    latest_market_values,
    on="player_id",
    how="left",
    validate="many_to_one",
)

current_players = current_players.rename(
    columns={
        "latest_market_value_in_eur": "previous_market_value_in_eur",
        "latest_valuation_date": "valuation_date",
    }
)

In [120]:
# fallback position for players without Transfermarkt static data.
current_players["pos"] = (
    current_players["pos"].str.split(",").str[0].str.strip()
)

position_map = {
    "GK": "Goalkeeper",
    "DF": "Defender",
    "MF": "Midfield",
    "FW": "Attack",
}

current_players["position"] = (
    current_players["position"]
    .fillna(current_players["pos"].map(position_map))
)

In [121]:
print("FBref rows:", len(fbref_current))
print("Rows after TM enrichment:", len(current_players))
print("Missing TM mapping:", current_players["player_id"].isna().sum())
print(
    "Missing previous value:",
    current_players["previous_market_value_in_eur"].isna().sum(),
)

assert len(current_players) == len(fbref_current)
assert current_players["competition_id"].notna().all()


FBref rows: 2105
Rows after TM enrichment: 2105
Missing TM mapping: 294
Missing previous value: 350


### 7. Collapse to One Player-Season Row

Players appearing for multiple clubs are aggregated across clubs. The primary club is the club with the most minutes, matching the historical modeling logic.

In [122]:
# get primary club
primary_rows = (
    current_players
    .sort_values("current_minutes", ascending=False)
    .drop_duplicates("player_code_fb")
    .copy()
)

# aggregate season totals across clubs
player_totals = (
    current_players
    .groupby("player_code_fb", as_index=False)
    .agg(
        current_games=("current_games", "sum"),
        current_minutes=("current_minutes", "sum"),
        current_90s=("current_90s", "sum"),
        current_goals=("current_goals", "sum"),
        current_assists=("current_assists", "sum"),
        current_yellow_cards=("current_yellow_cards", "sum"),
        current_red_cards=("current_red_cards", "sum"),
        clubs_played_for_season=("team", "nunique"),
    )
)

gk_clean_sheets = (
    current_players
    .groupby("player_code_fb")["current_clean_sheets_gk"]
    .sum(min_count=1)
    .reset_index()
)

replace_cols = [
    "current_games",
    "current_minutes",
    "current_90s",
    "current_goals",
    "current_assists",
    "current_yellow_cards",
    "current_red_cards",
    "current_clean_sheets_gk",
]

current_players = (
    primary_rows
    .drop(columns=replace_cols)
    .merge(
        player_totals,
        on="player_code_fb",
        how="left",
        validate="one_to_one",
    )
    .merge(
        gk_clean_sheets,
        on="player_code_fb",
        how="left",
        validate="one_to_one",
    )
)

In [123]:
# FBref rates can be kept directly for one-club players
# recalculate only where multiple club rows were combined
multi_club = current_players["clubs_played_for_season"] > 1
denom_90 = current_players.loc[multi_club, "current_90s"].replace(0, np.nan)

current_players.loc[multi_club, "current_goals_per_90"] = (
    current_players.loc[multi_club, "current_goals"] / denom_90
)

current_players.loc[multi_club, "current_assists_per_90"] = (
    current_players.loc[multi_club, "current_assists"] / denom_90
)

current_players.loc[multi_club, "current_goal_contributions_per_90"] = (
    (
        current_players.loc[multi_club, "current_goals"]
        + current_players.loc[multi_club, "current_assists"]
    )
    / denom_90
)

assert current_players["player_code_fb"].is_unique

print("Player-season rows:", len(current_players))
print("Multi-club players:", multi_club.sum())

Player-season rows: 2070
Multi-club players: 35


### 8. End-of-season Trajectory Calculation

The model was trained on completed seasons. Current totals are therefore extrapolated to a full league season using the primary club's current number of completed matches. /90 performance rates remain unchanged.

In [124]:
# build end-of-season projections

projected_players = current_players.copy()

projected_players["full_season_games"] = (
    projected_players["league"].map(LEAGUE_GAMES)
)

projected_players["projection_factor"] = (
    projected_players["full_season_games"]
    / projected_players["club_games_played"].replace(0, np.nan)
)

projected_players["projected_games"] = (
    projected_players["current_games"]
    * projected_players["projection_factor"]
).clip(upper=projected_players["full_season_games"])

projected_players["projected_minutes"] = (
    projected_players["current_minutes"]
    * projected_players["projection_factor"]
).clip(upper=projected_players["full_season_games"] * 90)

for current_col, projected_col in {
    "current_goals": "projected_goals",
    "current_assists": "projected_assists",
    "current_yellow_cards": "projected_yellow_cards",
    "current_red_cards": "projected_red_cards",
}.items():
    projected_players[projected_col] = (
        projected_players[current_col]
        * projected_players["projection_factor"]
    )

projected_players["projected_club_goals_scored"] = (
    projected_players["club_goals_scored"]
    * projected_players["projection_factor"]
)

projected_players["projected_club_goals_conceded"] = (
    projected_players["club_goals_conceded"]
    * projected_players["projection_factor"]
)

# Exact current GK clean-sheet rate where available.
# For outfield players, approximate with the primary club's clean-sheet rate.
gk_rate = (
    projected_players["current_clean_sheets_gk"]
    / projected_players["current_games"].replace(0, np.nan)
)

projected_players["clean_sheet_rate"] = np.where(
    projected_players["position"].eq("Goalkeeper")
    & projected_players["current_clean_sheets_gk"].notna(),
    gk_rate,
    projected_players["club_clean_sheet_rate"],
)

projected_players["total_clean_sheets"] = (
    projected_players["projected_games"]
    * projected_players["clean_sheet_rate"]
)


### 9. Add Age, Valuation and Reliable Transfer-History

FBref provides age as years-days plus birth year. These are used with the fixed snapshot date to reconstruct the date of birth and calculate age at the target valuation date.

Transfermarkt's 2026 summer transfer data is incomplete in the local source. We therefore keep only reliable pre-summer historical transfer counts and intentionally leave current-season transfer timing features missing for the model pipeline to impute.

In [125]:
age_parts = projected_players["age"].astype("string").str.extract(
    r"(?P<age_years>\d+)-(?P<age_days>\d+)"
)

projected_players["age_years"] = pd.to_numeric(
    age_parts["age_years"], errors="coerce"
)

projected_players["age_days"] = pd.to_numeric(
    age_parts["age_days"], errors="coerce"
)

projected_players["born"] = pd.to_numeric(
    projected_players["born"], errors="coerce"
)


def reconstruct_dob(row):
    if (
        pd.isna(row["age_years"])
        or pd.isna(row["age_days"])
        or pd.isna(row["born"])
    ):
        return pd.NaT

    last_birthday = (
        SNAPSHOT_DATE
        - pd.Timedelta(days=int(row["age_days"]))
    )

    return pd.Timestamp(
        year=int(row["born"]),
        month=last_birthday.month,
        day=last_birthday.day,
    )


projected_players["date_of_birth"] = projected_players.apply(
    reconstruct_dob,
    axis=1,
)


def age_on_date(dob, target_date):
    if pd.isna(dob):
        return pd.NA

    return (
        target_date.year
        - dob.year
        - (
            (target_date.month, target_date.day)
            < (dob.month, dob.day)
        )
    )


projected_players["age_at_valuation"] = (
    projected_players["date_of_birth"]
    .apply(lambda dob: age_on_date(dob, TARGET_VALUATION_DATE))
    .astype("Int64")
)

projected_players["valuation_date"] = pd.to_datetime(
    projected_players["valuation_date"],
    errors="coerce",
)

projected_players["log_previous_market_value"] = np.log1p(
    projected_players["previous_market_value_in_eur"]
)

projected_players["days_since_previous_valuation"] = (
    TARGET_VALUATION_DATE
    - projected_players["valuation_date"]
).dt.days


In [126]:
transfers = pd.read_csv("../data/raw/transfers.csv")[
    ["player_id", "transfer_date"]
]

transfers["player_id"] = pd.to_numeric(
    transfers["player_id"], errors="coerce"
).astype("Int64")

transfers["transfer_date"] = pd.to_datetime(
    transfers["transfer_date"],
    errors="coerce",
)

# avoid the incomplete 2026 summer window
RELIABLE_TRANSFER_CUTOFF = pd.Timestamp("2026-06-01")

historical_transfers = transfers.loc[
    transfers["transfer_date"] < RELIABLE_TRANSFER_CUTOFF
]

transfer_history = (
    historical_transfers
    .groupby("player_id", as_index=False)
    .agg(total_previous_transfers=("transfer_date", "count"))
)

projected_players = projected_players.merge(
    transfer_history,
    on="player_id",
    how="left",
    validate="many_to_one",
)

mapped_no_history = (
    projected_players["player_id"].notna()
    & projected_players["total_previous_transfers"].isna()
)

projected_players.loc[
    mapped_no_history,
    "total_previous_transfers",
] = 0

projected_players["has_previous_transfer"] = np.where(
    projected_players["player_id"].isna(),
    np.nan,
    (projected_players["total_previous_transfers"] > 0).astype(float),
)

# current-season transfer data is not reliable enough to encode as zero.
projected_players["transfers_during_season"] = np.nan
projected_players["days_since_last_transfer"] = np.nan


### 10. Build the Model-Ready Feature Schema

Projected totals are mapped to the same feature names used during model training. The historical 900-minute rule is applied to /90 features.

In [127]:
# get production models metadata

import json

METADATA_PATH = "../models/production_metadata.json"

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

FEATURES_WITHOUT_HISTORY = metadata["features_without_history"]
FEATURES_WITH_HISTORY = metadata["features_with_history"]

In [128]:
projected_players["season"] = SEASON

# player season totals
projected_players["total_games_played_season"] = projected_players["projected_games"]
projected_players["total_minutes_played"] = projected_players["projected_minutes"]
projected_players["total_goals"] = projected_players["projected_goals"]
projected_players["total_assists"] = projected_players["projected_assists"]
projected_players["total_yellow_cards"] = projected_players["projected_yellow_cards"]
projected_players["total_red_cards"] = projected_players["projected_red_cards"]

# FBref already provides attacking /90 rates
projected_players["goals_per_90_min"] = projected_players["current_goals_per_90"]
projected_players["assists_per_90_min"] = projected_players["current_assists_per_90"]
projected_players["goal_contributions_per_90_min"] = (
    projected_players["current_goal_contributions_per_90"]
)

# card rates are not provided directly
current_90s = projected_players["current_90s"].replace(0, np.nan)

projected_players["yellow_cards_per_90_min"] = (
    projected_players["current_yellow_cards"] / current_90s
)

projected_players["red_cards_per_90_min"] = (
    projected_players["current_red_cards"] / current_90s
)

# projected club totals
projected_players["club_goals_scored"] = (
    projected_players["projected_club_goals_scored"]
)

projected_players["club_goals_conceded"] = (
    projected_players["projected_club_goals_conceded"]
)

projected_players["current_club_games_played"] = (
    projected_players["club_games_played"]
)

projected_players["club_games_played"] = (
    projected_players["full_season_games"]
)

In [129]:
# count features should be whole numbers
COUNT_FEATURES = [
    "total_games_played_season",
    "total_minutes_played",
    "total_goals",
    "total_assists",
    "total_yellow_cards",
    "total_red_cards",
    "total_clean_sheets",
    "clubs_played_for_season",
    "club_goals_scored",
    "club_goals_conceded",
    "club_games_played",
    "total_previous_transfers",
    "transfers_during_season",
    "has_previous_transfer",
]

for col in COUNT_FEATURES:
    projected_players[col] = (
        projected_players[col]
        .round()
        .astype("Int64")
    )

In [130]:
required_features = list(
    dict.fromkeys(
        FEATURES_WITHOUT_HISTORY
        + FEATURES_WITH_HISTORY
    )
)

missing_features = [
    feature
    for feature in required_features
    if feature not in projected_players.columns
]

assert not missing_features, (
    f"Missing production features: {missing_features}"
)

assert projected_players["player_code_fb"].is_unique
assert projected_players["competition_id"].notna().all()

print("Final player rows:", len(projected_players))
print("Production features:", len(required_features))

print(
    "With-history route:",
    projected_players["previous_market_value_in_eur"].notna().sum()
)

print(
    "Without-history route:",
    projected_players["previous_market_value_in_eur"].isna().sum()
)

Final player rows: 2070
Production features: 31
With-history route: 1726
Without-history route: 344


### 11. Save Current-Season Inference Dataset

Keep identifiers/context plus every feature required by either production model.

In [131]:
# save inference dataset
ID_COLUMNS = [
    "player",
    "player_code_fb",
    "player_id",
    "team",
    "league",
    "image_url",
    "previous_market_value_in_eur",
    "valuation_date",
    "country_of_citizenship",
]

CURRENT_COLUMNS = [
    "current_games",
    "current_minutes",
    "current_goals",
    "current_assists",
    "current_yellow_cards",
    "current_red_cards",
    "current_club_games_played"
]

PROJECTION_COLUMNS = [
    "full_season_games",
    "projection_factor",
    "projected_games",
    "projected_minutes",
    "projected_goals",
    "projected_assists",
    "projected_yellow_cards",
    "projected_red_cards",
    "projected_club_goals_scored",
    "projected_club_goals_conceded",
]

MODEL_FEATURES = list(
    dict.fromkeys(
        FEATURES_WITHOUT_HISTORY
        + FEATURES_WITH_HISTORY
    )
)

current_season_inference = projected_players[
    ID_COLUMNS
    + CURRENT_COLUMNS
    + PROJECTION_COLUMNS
    + MODEL_FEATURES
].copy()

In [133]:
OUTPUT_PATH = "../data/processed/current_season_inference_2026_27.csv"

current_season_inference.to_csv(
    OUTPUT_PATH,
    index=False
)

In [134]:
display(current_season_inference)

,player,player_code_fb,player_id,team,league,image_url,previous_market_value_in_eur,valuation_date,country_of_citizenship,current_games,...,club_goals_scored,club_goals_conceded,club_games_played,avg_points_per_game,total_previous_transfers,transfers_during_season,has_previous_transfer,days_since_last_transfer,log_previous_market_value,days_since_previous_valuation
0,Marko Dmitrović,marko-dmitrovic,94308.0,Espanyol,ESP-La Liga,https://img.a.transfermarkt.technology/portrai...,700000.0,2026-06-05,Serbia,7,...,54,54,38,1.000000,13,<NA>,1,NaN,13.458837,361.0
1,Ionuț Radu,ionut-radu,303657.0,Celta Vigo,ESP-La Liga,https://img.a.transfermarkt.technology/portrai...,9000000.0,2026-06-05,Romania,7,...,43,33,38,1.000000,23,<NA>,1,NaN,16.012735,361.0
2,Leo Román,leo-roman,562742.0,Dep. A Coruña,ESP-La Liga,https://img.a.transfermarkt.technology/portrai...,6000000.0,2026-06-05,Spain,7,...,54,43,38,1.428571,7,<NA>,1,NaN,15.607270,361.0
3,Antonio Sivera,antonio-sivera,298197.0,Alavés,ESP-La Liga,https://img.a.transfermarkt.technology/portrai...,6000000.0,2026-06-05,Spain,7,...,60,33,38,1.571429,6,<NA>,1,NaN,15.607270,361.0
4,Stole Dimitrievski,stole-dimitrievski,165467.0,Valencia,ESP-La Liga,https://img.a.transfermarkt.technology/portrai...,2000000.0,2026-06-05,North Macedonia,7,...,22,71,38,0.571429,12,<NA>,1,NaN,14.508658,361.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2065,Joël Latibeaudière,joel-latibeaudiere,412823.0,Coventry City,ENG-Premier League,https://img.a.transfermarkt.technology/portrai...,3000000.0,2026-05-28,Jamaica,1,...,8,76,38,0.600000,6,<NA>,1,NaN,14.914123,369.0
2066,Hakim Dongopandji,hakim-dongopandji,<NA>,Le Havre,FRA-Ligue 1,NaN,NaN,NaT,NaN,1,...,27,48,34,0.400000,<NA>,<NA>,<NA>,NaN,NaN,NaN
2067,Mattia Perin,mattia-perin,110923.0,Juventus,ITA-Serie A,https://img.a.transfermarkt.technology/portrai...,2000000.0,2026-05-29,Italy,1,...,61,30,38,2.000000,9,<NA>,1,NaN,14.508658,368.0
2068,Omri Gandelman,omri-gandelman,780548.0,Lecce,ITA-Serie A,https://img.a.transfermarkt.technology/portrai...,3000000.0,2026-05-29,Israel,1,...,38,76,38,1.200000,5,<NA>,1,NaN,14.914123,368.0
